# USPS digits depth / ARD / 1-NN (Damianou & Lawrence 2013, §4.3 / figs. 6–8)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/PyDeepGP/blob/main/examples/damianou2013/notebooks/03_usps_digits.ipynb)

**Paper:** [Damianou, A. and Lawrence, N. D. (2013). *Deep Gaussian Processes*. In *AISTATS* (pp. 207–215).](https://proceedings.mlr.press/v31/damianou13a.html)

Subsample 50 examples of each digit $\{0,1,6\}$ from USPS ($16\times16$). Train unsupervised DeepGPs of increasing depth and evaluate (i) the variational bound and (ii) leave-one-out 1-NN error in the top latent features. The paper finds both the bound and nearest-neighbour quality improve with depth, with a 5-layer model nearly separating the classes in a low-dimensional top space.


In [ ]:
# Install dependencies when running on Google Colab
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    pkgs = ["numpy>=1.7", "scipy>=0.12", "matplotlib", "GPy>=1.15"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/SheffieldML/PyDeepGP.git@main",
    ])
    print("Installed GPy and PyDeepGP for Colab.")
else:
    print("Local environment — ensure PyDeepGP and GPy are importable.")


In [ ]:
# Set QUICK=True for a short smoke run (fewer iterations / smaller models).
# Set QUICK=False for a paper-scale demonstration (slower).
QUICK = True
SEED = 42

DEPTHS = [1, 2] if QUICK else [1, 2, 3, 4, 5]
PER_CLASS = 20 if QUICK else 50


In [ ]:
import gzip
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import GPy
import deepgp

np.random.seed(SEED)
DIGITS = (0, 1, 6)
ESL_URL = "https://web.stanford.edu/~hastie/ElemStatLearn/datasets/zip.train.gz"
CACHE = os.path.join("/tmp", "pydeepgp_usps_zip.train.gz")

def download_usps(path=CACHE):
    if os.path.isfile(path) and os.path.getsize(path) > 0:
        return path
    print("Downloading USPS zip.train …")
    req = urllib.request.Request(ESL_URL, headers={"User-Agent": "PyDeepGP-notebooks"})
    with urllib.request.urlopen(req, timeout=120) as resp, open(path, "wb") as out:
        out.write(resp.read())
    return path

def load_usps(path):
    rows = []
    with gzip.open(path, "rt") as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                rows.append([float(x) for x in parts])
    data = np.asarray(rows)
    return data[:, 1:], data[:, 0].astype(int)

def subsample(Y, labels, per_class=50, seed=42):
    rng = np.random.RandomState(seed)
    Ys, Ls = [], []
    for d in DIGITS:
        idx = np.where(labels == d)[0]
        choose = rng.choice(idx, size=per_class, replace=False)
        Ys.append(Y[choose]); Ls.append(np.full(per_class, d))
    return np.vstack(Ys), np.concatenate(Ls)

def rbf_bias(q):
    return GPy.kern.RBF(q, ARD=True) + GPy.kern.Bias(q)

def ard_weights(kern):
    ls = np.asarray(kern.rbf.lengthscale).ravel() if hasattr(kern, "rbf") else np.asarray(kern.lengthscale).ravel()
    return 1.0 / np.maximum(ls, 1e-12)

def nn_error(X, labels):
    sq = np.sum(X * X, axis=1, keepdims=True)
    d2 = sq + sq.T - 2 * X @ X.T
    np.fill_diagonal(d2, np.inf)
    nn = np.argmin(d2, axis=1)
    mistakes = int(np.sum(labels[nn] != labels))
    return mistakes, mistakes / len(labels)

def depth_dims(depth, data_dim, quick):
    widths = {1: [8], 2: [8, 5], 3: [8, 6, 4]} if quick else {
        1: [12], 2: [12, 8], 3: [12, 10, 6], 4: [12, 10, 8, 5], 5: [12, 10, 8, 6, 4]
    }
    return [data_dim] + widths[depth]


In [ ]:
path = download_usps()
Y_all, lab_all = load_usps(path)
Y, labels = subsample(Y_all, lab_all, per_class=PER_CLASS, seed=SEED)
Y = (Y - Y.mean(0)) / (Y.std() + 1e-12)
print("Subset", Y.shape, "classes", sorted(set(labels.tolist())))

# Show a few digits
fig, axes = plt.subplots(1, 6, figsize=(8, 1.6))
for ax, i in zip(axes, range(6)):
    ax.imshow(Y[i].reshape(16, 16), cmap="gray")
    ax.set_title(str(labels[i]))
    ax.axis("off")
plt.suptitle("USPS subset samples")
plt.show()


## Fit depths and score bound / 1-NN


In [ ]:
from GPy.core.parameterization.variational import NormalPosterior, VariationalPosterior

def out_var(y):
    if isinstance(y, (NormalPosterior, VariationalPosterior)):
        return float(np.asarray(y.mean).var())
    return float(np.asarray(y).var())

num_inducing = 20 if QUICK else 40
iters_fixed, iters_free = (40, 100) if QUICK else (200, 2000)

results, models = [], {}
for depth in DEPTHS:
    print(f"=== depth {depth} ===")
    dims = depth_dims(depth, Y.shape[1], QUICK)
    m = deepgp.DeepGP(
        dims, Y,
        kernels=[rbf_bias(q) for q in dims[1:]],
        num_inducing=num_inducing,
        back_constraint=False,
    )
    for layer in m.layers:
        layer.Gaussian_noise.variance = out_var(layer.Y) * 0.01
        layer.Gaussian_noise.variance.fix()
    m.optimize(max_iters=iters_fixed, messages=False)
    for layer in m.layers:
        layer.Gaussian_noise.variance.unfix()
    m.optimize(max_iters=iters_free, messages=False)
    bound = float(m.log_likelihood())
    Xtop = np.asarray(m.layers[-1].X.mean)
    mistakes, rate = nn_error(Xtop, labels)
    print(f"bound={bound:.3f}  1-NN mistakes={mistakes}/{Y.shape[0]} ({rate:.3%})")
    results.append((depth, bound, mistakes, rate))
    models[depth] = m

print("\ndepth  bound      1-NN errors")
for depth, bound, mistakes, rate in results:
    print(f"{depth:5d}  {bound:9.3f}  {mistakes:3d} ({rate:.3%})")


## Visualisations (figs. 6–7 style)


In [ ]:
deepest = max(DEPTHS)
m = models[deepest]
Xtop = np.asarray(m.layers[-1].X.mean)
w = ard_weights(m.layers[-1].kern)
dims = list(np.argsort(-w)[:2])
colors = {0: "C0", 1: "C1", 6: "C2"}
markers = {0: "o", 1: "s", 6: "^"}

fig, ax = plt.subplots(figsize=(4.5, 4.5))
for d in DIGITS:
    mask = labels == d
    ax.scatter(Xtop[mask, dims[0]], Xtop[mask, dims[1]],
               c=colors[d], marker=markers[d], s=28, label=str(d))
ax.set_xlabel(f"top dim {dims[0]}")
ax.set_ylabel(f"top dim {dims[1]}")
ax.set_title(f"Top latent (depth {deepest})")
ax.legend()
plt.show()

fig, axes = plt.subplots(1, len(m.layers), figsize=(3.0 * len(m.layers), 2.8))
if len(m.layers) == 1:
    axes = [axes]
for i, layer in enumerate(m.layers):
    ww = ard_weights(layer.kern)
    axes[i].bar(np.arange(1, len(ww) + 1), ww)
    axes[i].set_title(f"layer {i} ARD")
fig.suptitle("ARD weights (1/lengthscale) — fig. 6 style")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot([r[0] for r in results], [r[1] for r in results], "o-", label="variational bound")
ax.set_xlabel("depth"); ax.set_ylabel("bound")
ax2 = ax.twinx()
ax2.plot([r[0] for r in results], [r[2] for r in results], "s--", color="C1", label="1-NN mistakes")
ax2.set_ylabel("1-NN mistakes")
ax.set_title("Depth selection via bound / 1-NN")
plt.show()


## References

- Damianou, A. and Lawrence, N. D. (2013). *Deep Gaussian Processes*. In *AISTATS* (pp. 207–215). — [PDF](https://proceedings.mlr.press/v31/damianou13a.html)
- USPS via ElemStatLearn `zip.train`
- Script counterpart: `examples/damianou2013/usps_digits.py`
